<a href="https://colab.research.google.com/github/gitbidec6/Machine-Learning-Works/blob/main/Tensile_Strength.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
pip install numpy pandas statsmodels matplotlib seaborn scipy patsy

In [ ]:
# mixture_analysis.py  (or paste into a Jupyter/Colab cell)
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt
import seaborn as sns
from itertools import combinations
from statsmodels.stats.outliers_influence import variance_inflation_factor
from scipy.optimize import minimize

# ---------- 1) Load data ----------
data = {
    "Polycaprolactone": [2.4, 2.4, 2.4, 2.1, 2.1, 2.1, 2.64, 2.52, 2.4, 0, 0, 0],
    "Polylactic_Acid": [0.54, 0.42, 0.3, 0.54, 0.42, 0.3, 0, 0, 0, 2.64, 2.52, 2.4],
    "Hydroxyapatite": [0, 0, 0, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3, 0.3],
    "Gelatin": [0.06, 0.18, 0.3, 0.06, 0.18, 0.3, 0.06, 0.18, 0.3, 0.06, 0.18, 0.3],
    "Tensile_Strength": [3.23, 4.19, 2.61, 4.41, 3.12, 4.31, 3.74, 3.91, 3.86, 9.44, 8.01, 5.92]
}
df = pd.DataFrame(data)
print("Raw data:\n", df)

# ---------- 2) Convert to proportions (mixture) ----------
# If your components are amounts per run, convert to proportions per run:
component_cols = ["Polycaprolactone", "Polylactic_Acid", "Hydroxyapatite", "Gelatin"]
df['sum_components'] = df[component_cols].sum(axis=1)
# Avoid division by zero — if sum == 0 (shouldn't happen), drop or fix row
if (df['sum_components'] == 0).any():
    raise ValueError("One or more rows have zero total composition: cannot convert to proportions.")
# create proportion columns
for c in component_cols:
    df[c + "_p"] = df[c] / df['sum_components']

prop_cols = [c + "_p" for c in component_cols]
print("\nProportions (first rows):\n", df[prop_cols].head())

# ---------- 3) Build Scheffé models ----------
# 3a) Linear Scheffé model (no intercept): y = sum(beta_i * x_i)
X_lin = df[prop_cols].copy()
y = df['Tensile_Strength']

# Fit linear model without intercept
model_lin = sm.OLS(y, X_lin).fit()
print("\nLinear Scheffé model summary:\n", model_lin.summary())

# 3b) Quadratic Scheffé (pairwise terms)
X_quad = df[prop_cols].copy()
pairs = list(combinations(prop_cols, 2))
for a, b in pairs:
    X_quad[f"{a}*{b}"] = X_quad[a] * X_quad[b]
# No intercept for Scheffé mixture model
model_quad = sm.OLS(y, X_quad).fit()
print("\nQuadratic Scheffé model summary:\n", model_quad.summary())

# Save model outputs
with open("model_summaries.txt", "w") as f:
    f.write("LINEAR SCHEFFÉ\n")
    f.write(model_lin.summary().as_text())
    f.write("\n\nQUADRATIC SCHEFFÉ\n")
    f.write(model_quad.summary().as_text())

# ---------- 4) Diagnostics ----------
# Predictions & residuals
df['pred_lin'] = model_lin.predict(X_lin)
df['resid_lin'] = df['Tensile_Strength'] - df['pred_lin']

df['pred_quad'] = model_quad.predict(X_quad)
df['resid_quad'] = df['Tensile_Strength'] - df['pred_quad']

# Actual vs Predicted plot (quadratic)
plt.figure(figsize=(6,5))
plt.scatter(df['Tensile_Strength'], df['pred_quad'], s=50)
mn = min(df['Tensile_Strength'].min(), df['pred_quad'].min())
mx = max(df['Tensile_Strength'].max(), df['pred_quad'].max())
plt.plot([mn,mx],[mn,mx], linestyle='--')
plt.xlabel("Actual Tensile Strength")
plt.ylabel("Predicted (Quadratic Scheffé)")
plt.title("Actual vs Predicted")
plt.tight_layout()
plt.savefig("actual_vs_predicted_quad.png", dpi=200)
plt.close()

# Residuals vs Predicted
plt.figure(figsize=(6,4))
plt.scatter(df['pred_quad'], df['resid_quad'], s=50)
plt.axhline(0, color='k', linestyle='--')
plt.xlabel("Predicted (Quadratic)")
plt.ylabel("Residuals")
plt.title("Residuals vs Predicted")
plt.tight_layout()
plt.savefig("residuals_vs_predicted_quad.png", dpi=200)
plt.close()

# Residual histogram + QQ
plt.figure(figsize=(6,4))
sns.histplot(df['resid_quad'], kde=True)
plt.title("Residuals (Quadratic)")
plt.tight_layout()
plt.savefig("residuals_hist_quad.png", dpi=200)
plt.close()

# ---------- 5) VIF (note: interpret carefully for mixtures) ----------
# VIF on the X_quad matrix (add a small regularization if perfect multicollinearity)
X_vif = X_quad.copy()
# Add tiny noise if singular to avoid divide-by-zero
X_vif += 0.0
vif_data = []
for i, col in enumerate(X_vif.columns):
    try:
        vif = variance_inflation_factor(X_vif.values, i)
    except Exception:
        vif = np.inf
    vif_data.append((col, vif))
vif_df = pd.DataFrame(vif_data, columns=['variable','VIF']).sort_values('VIF', ascending=False)
vif_df.to_csv("vif.csv", index=False)
print("\nVIFs:\n", vif_df)

# ---------- 6) Model comparison ----------
print("\nModel comparison:")
print("Linear R2:", model_lin.rsquared, "Adj R2:", model_lin.rsquared_adj)
print("Quadratic R2:", model_quad.rsquared, "Adj R2:", model_quad.rsquared_adj)
print("Quadratic AIC/BIC:", model_quad.aic, model_quad.bic)

# ---------- 7) Optimization / grid search under mixture constraint ----------
# We'll search over the simplex (proportions sum to 1, all >= 0)
# Use grid sampling (simple and robust for 4 components); for finer search use constrained optimizer.

def predict_quad_from_props(props, model, prop_cols):
    # props: array-like of length 4 (proportions in same order as prop_cols)
    row = dict(zip(prop_cols, props))
    # construct pairwise features
    for a,b in combinations(prop_cols, 2):
        row[f"{a}*{b}"] = row[a] * row[b]
    # order columns same as X_quad
    Xrow = np.array([row[c] for c in X_quad.columns], dtype=float)
    return float(np.dot(Xrow, model.params))

# Grid search: simple barycentric sampling
grid_list = []
n_steps = 20  # increase for finer search (costs more compute)
for i in range(n_steps+1):
    for j in range(n_steps+1 - i):
        for k in range(n_steps+1 - i - j):
            l = n_steps - i - j - k
            props = np.array([i,j,k,l], dtype=float) / n_steps
            pred = predict_quad_from_props(props, model_quad, prop_cols)
            grid_list.append((props, pred))
# collect top few
grid_df = pd.DataFrame([{"P":g[0][0], "PLA":g[0][1], "HA":g[0][2], "Gel":g[0][3], "pred":g[1]} for g in grid_list])
top = grid_df.sort_values("pred", ascending=False).head(10)
top.to_csv("top_predicted_compositions.csv", index=False)
print("\nTop predicted compositions (grid search):\n", top.head())

# ---------- 8) Optional: constrained continuous optimization (refinement) ----------
# Use quadratic model linear-in-params so objective is simple; we can run constrained optimization
def neg_pred_to_min(x):
    # x are proportions length-4, must sum to 1
    return -predict_quad_from_props(x, model_quad, prop_cols)

# initial guess = top grid
x0 = top[['P','PLA','HA','Gel']].iloc[0].values
cons = ({'type':'eq', 'fun': lambda x: np.sum(x)-1.0})
bnds = tuple((0.0,1.0) for _ in range(4))
res = minimize(neg_pred_to_min, x0, method='SLSQP', bounds=bnds, constraints=cons)
if res.success:
    opt_props = res.x
    opt_pred = -res.fun
    print("\nOptimizer result (refined):")
    print("Proportions:", dict(zip(prop_cols, opt_props)))
    print("Predicted tensile strength:", opt_pred)
else:
    print("\nOptimizer failed or not improved. status:", res.message)

# ---------- 9) Save data outputs ----------
df.to_csv("data_with_predictions.csv", index=False)
grid_df.to_csv("full_grid_predictions.csv", index=False)

print("\nSaved plots: actual_vs_predicted_quad.png, residuals_vs_predicted_quad.png, residuals_hist_quad.png")
print("Saved CSVs: model_summaries.txt, vif.csv, top_predicted_compositions.csv, data_with_predictions.csv, full_grid_predictions.csv")


Raw data:
     Polycaprolactone  Polylactic_Acid  Hydroxyapatite  Gelatin  \
0               2.40             0.54             0.0     0.06   
1               2.40             0.42             0.0     0.18   
2               2.40             0.30             0.0     0.30   
3               2.10             0.54             0.3     0.06   
4               2.10             0.42             0.3     0.18   
5               2.10             0.30             0.3     0.30   
6               2.64             0.00             0.3     0.06   
7               2.52             0.00             0.3     0.18   
8               2.40             0.00             0.3     0.30   
9               0.00             2.64             0.3     0.06   
10              0.00             2.52             0.3     0.18   
11              0.00             2.40             0.3     0.30   

    Tensile_Strength  
0               3.23  
1               4.19  
2               2.61  
3               4.41  
4              

/usr/local/lib/python3.12/dist-packages/statsmodels/stats/outliers_influence.py:197: RuntimeWarning: divide by zero encountered in scalar divide
  vif = 1. / (1. - r_squared_i)



VIFs:
                                variable          VIF
0                    Polycaprolactone_p          inf
1                     Polylactic_Acid_p          inf
2                      Hydroxyapatite_p          inf
3                             Gelatin_p          inf
5   Polycaprolactone_p*Hydroxyapatite_p          inf
7    Polylactic_Acid_p*Hydroxyapatite_p          inf
9            Hydroxyapatite_p*Gelatin_p          inf
6          Polycaprolactone_p*Gelatin_p  1544.060677
8           Polylactic_Acid_p*Gelatin_p   959.050630
4  Polycaprolactone_p*Polylactic_Acid_p     2.076813

Model comparison:
Linear R2: 0.8589086011679423 Adj R2: 0.8059993266059207
Quadratic R2: 0.9532278215781087 Adj R2: 0.8713765093397989
Quadratic AIC/BIC: 29.604520648442513 33.483773846746516

Top predicted compositions (grid search):
         P   PLA   HA   Gel       pred
230  0.00  1.00  0.0  0.00  18.340386
440  0.05  0.95  0.0  0.00  17.182349
630  0.10  0.90  0.0  0.00  16.056333
228  0.00  0.95  0.0